
# Cloud Storage (GCS) Hands-On Lab — Google Cloud SDK
### GCP Training Program — Day 1, Module 1: Storage & Ingestion

## Problem Statement

A growing e-commerce company needs somewhere to put files that don't belong in a database —
product images, uploaded invoice PDFs, nightly export CSVs headed for BigQuery, ML model
checkpoints, log archives kept for compliance. None of these are rows in a table; they're
**objects**, and they need to be stored cheaply, organized predictably, secured precisely, and
served efficiently. This notebook builds up exactly those capabilities, one at a time, against a
single working bucket:

1. **"Where do these files actually live, and how do I organize them?"** — buckets, uploads,
   listing, copy/rename/delete *(Sections 4-7)*
2. **"How do I know what a file actually is without downloading it?"** — object metadata
   *(Section 8)*
3. **"Some of this data is accessed constantly, some almost never — should it all cost the same to
   store?"** — storage classes and lifecycle rules *(Sections 9-10)*
4. **"Someone overwrote a file — can we get the old version back?"** — object versioning
   *(Section 11)*
5. **"Who should be allowed to read or write this, and how do we prove it later?"** — ACLs, IAM,
   signed URLs, CORS *(Sections 12-14)*
6. **"We generated a file in pieces — can we combine them without downloading and re-uploading?"**
   — object composition *(Section 15)*

This is the **core/foundations version** of this lab — covering everything through object
composition. A companion **Full** version of this same notebook additionally covers retries,
batch requests, and Transfer Manager for parallel/chunked file transfer (Sections 16-20), for a
session with more time available.

Everything runs through the **`google-cloud-storage` Python client library only** — no CLI, no
`subprocess`, nothing shelled out anywhere in this notebook. Every operation includes a
**🖥️ Check it in the console** pointer showing exactly where the same thing is visible in the
Cloud Console, so what the code just did isn't left as an abstract claim.

**Scope note:** this lab is scoped to **Cloud Storage** only. Spanner, Cloud SQL, Firestore, and
Bigtable are covered as separate hands-on labs elsewhere in Module 1.

**This notebook is fully self-contained.** Just run the cells top to bottom:
1. The **Authenticate** cell below will pop up a Google sign-in flow — sign in with the account
   that has access to your GCP project.
2. The **Configure** cell will *ask you* for your Project ID, region, and bucket name — nothing to
   hand-edit in code.
3. Everything after that runs against your own project automatically.

**Prerequisites (mention to the class before running):**
1. A GCP project with billing enabled.
2. IAM role on the project: `roles/storage.admin` (or equivalent) for the account you sign in with.
3. Python package `google-cloud-storage` (installed in a cell below).


## 1. Authenticate This Session
Sign in with the Google account that has access to your GCP project. This works whether you're
running in **Google Colab** (pops up an interactive sign-in) or a local/Vertex Workbench kernel
(falls back to the Cloud SDK's Application Default Credentials).


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures the gcloud CLI for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth login")
    print("  gcloud auth application-default login")

# Confirm which identity and project the Cloud SDK is using
!gcloud config list
!gcloud auth list


> 🖥️ **Check it in the console:** **IAM & Admin > IAM** → confirm the signed-in account shows a
> role covering Storage (e.g. **Storage Admin** or **Owner/Editor**) — the account this
> authentication step just set up is what every later console checkpoint in this notebook will be
> checking as.


## 2. Install & Import the Client Library

In [ ]:
!pip install --quiet google-cloud-storage

In [ ]:
from google.cloud import storage
from datetime import timedelta
import time

print("google-cloud-storage imported successfully")

## 3. Configure Your Project, Region & Bucket
Enter your own values when prompted below — nothing to hand-edit in code. The bucket name gets a
timestamp suffix so it's globally unique (bucket names are global across all of GCS).

In [ ]:
import time

PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your region [default: asia-south1]: ").strip()
if not REGION:
    REGION = "asia-south1"

_default_bucket = f"gcs-training-demo-{int(time.time())}"
BUCKET_NAME = input(f"Enter a bucket name [default: {_default_bucket}]: ").strip()
if not BUCKET_NAME:
    BUCKET_NAME = _default_bucket

!gcloud config set project {PROJECT_ID}

client = storage.Client(project=PROJECT_ID)
print("Client initialized for project:", client.project)
print("Target bucket name:", BUCKET_NAME)

## 4. Bucket Operations
### 4.1 Create a Bucket

In [ ]:
bucket = client.bucket(BUCKET_NAME)
bucket.storage_class = "STANDARD"

new_bucket = client.create_bucket(bucket, location=REGION)
print(f"Created gs://{new_bucket.name} in {new_bucket.location} ({new_bucket.storage_class})")

# gsutil / Cloud SDK equivalent:
# gsutil mb -p PROJECT_ID -l asia-south1 -c STANDARD gs://BUCKET_NAME


> 🖥️ **Check it in the console:** **Cloud Storage > Buckets** → your new bucket should now be
> listed, with its **Location** and **Default storage class** columns matching what was just
> printed — this is the single landing page for everything Sections 4-15 build on top of.


### 4.2 List All Buckets in the Project

In [ ]:
print("Buckets in project", PROJECT_ID, ":")
for b in client.list_buckets():
    print(" -", b.name)

# gsutil equivalent: gsutil ls


> 🖥️ **Check it in the console:** same **Cloud Storage > Buckets** page → every bucket printed
> above should appear in this list — a good moment to point out this is project-wide, not just
> the one bucket this notebook created.


### 4.3 Get Bucket Metadata

In [ ]:
bucket = client.get_bucket(BUCKET_NAME)

print("Name:              ", bucket.name)
print("Location:          ", bucket.location)
print("Storage class:     ", bucket.storage_class)
print("Created:           ", bucket.time_created)
print("Uniform bucket-level access:",
      bucket.iam_configuration.uniform_bucket_level_access_enabled)

# gsutil equivalent: gsutil ls -L -b gs://BUCKET_NAME


> 🖥️ **Check it in the console:** **Cloud Storage > Buckets > [your bucket] > Configuration** tab
> → shows the same location, storage class, and creation date fields just printed, laid out as a
> readable details page rather than a Python dict.


### 4.4 Update Bucket Configuration (labels + uniform access)

In [ ]:
bucket.labels = {"env": "training", "module": "gcs-lab"}
bucket.iam_configuration.uniform_bucket_level_access_enabled = True
bucket.patch()

print("Labels applied:", bucket.labels)
print("Uniform bucket-level access:", bucket.iam_configuration.uniform_bucket_level_access_enabled)


> 🖥️ **Check it in the console:** **Cloud Storage > Buckets > [your bucket] > Configuration** tab
> → the **Labels** field should show `env: training` and `module: gcs-lab`, and further down,
> **Access control** should now read **Uniform** — both changes made by this one cell, visible
> immediately without a page refresh.


## 5. Object Upload / Download
### 5.1 Upload a Local File

In [ ]:
with open("sample.txt", "w") as f:
    f.write("Hello from the GCP Cloud Storage hands-on lab!\n")

blob = bucket.blob("uploads/sample.txt")
blob.upload_from_filename("sample.txt")
print(f"Uploaded to gs://{BUCKET_NAME}/{blob.name}")

# gsutil equivalent:
# gsutil cp sample.txt gs://BUCKET_NAME/uploads/sample.txt


> 🖥️ **Check it in the console:** **Cloud Storage > Buckets > [your bucket] > Objects** tab →
> navigate into the `uploads/` folder → `sample.txt` should be listed with its size and a
> **Public access** column (should read "Not public" at this point).


### 5.2 Upload From In-Memory Data (no local file needed)

In [ ]:
blob2 = bucket.blob("uploads/inmemory.json")
blob2.upload_from_string('{"lab": "gcs", "day": 1}', content_type="application/json")
print("Uploaded in-memory object:", blob2.name)


> 🖥️ **Check it in the console:** same **Objects** tab → `uploads/inmemory.json` should now also
> be listed → click it → the **Object details** panel shows **Content-Type: application/json**,
> confirming the type we set in code without ever touching a local file.


### 5.3 Download an Object to a Local File

In [ ]:
blob = bucket.blob("uploads/sample.txt")
blob.download_to_filename("sample_downloaded.txt")

with open("sample_downloaded.txt") as f:
    print("Downloaded content ->", f.read())

# gsutil equivalent:
# gsutil cp gs://BUCKET_NAME/uploads/sample.txt sample_downloaded.txt

### 5.4 Download an Object Directly Into Memory

In [ ]:
content = bucket.blob("uploads/inmemory.json").download_as_text()
print("In-memory content:", content)


> 🖥️ **Check it in the console:** nothing new to check here — this confirms the console and the
> SDK are reading the exact same underlying bytes, just through two different access paths (a
> browser download vs. an in-memory API call).


## 6. Listing & Organizing Objects
### 6.1 List All Objects in the Bucket

In [ ]:
for b in bucket.list_blobs():
    print(b.name, "-", b.size, "bytes")


> 🖥️ **Check it in the console:** **Cloud Storage > Buckets > [your bucket] > Objects** tab (root
> view, no folder) → every object printed above should be visible here, with a **Size** column
> matching the bytes printed by this cell.


### 6.2 List Objects Under a Prefix (simulated "folder")

In [ ]:
for b in bucket.list_blobs(prefix="uploads/"):
    print(b.name)


> 🖥️ **Check it in the console:** click into the `uploads/` folder in the Objects tab → only
> objects under that prefix are shown — the console's folder view **is** prefix filtering, exactly
> like this `prefix="uploads/"` call, just rendered as clickable folders instead of a flat list.


### 6.3 List With a Delimiter (one-level folder view)
`delimiter="/"` makes the API return top-level "folders" (prefixes) separately from top-level objects — this is exactly how the Cloud Console file browser simulates folders, since GCS has no real directories.

In [ ]:
iterator = bucket.list_blobs(prefix="", delimiter="/")
blobs = list(iterator)   # must consume the iterator before .prefixes is populated

print("Objects at root:   ", [b.name for b in blobs])
print("Prefixes (folders):", list(iterator.prefixes))


> 🖥️ **Check it in the console:** the Objects tab's default view (breadcrumbs, folder icons) is
> built entirely on this `delimiter="/"` mechanism — there's no actual folder resource anywhere in
> GCS; the console is doing exactly the client-side grouping this cell just did manually.


## 7. Copy, Rename, Move & Delete Objects
### 7.1 Copy an Object

In [ ]:
source_blob = bucket.blob("uploads/sample.txt")
copied_blob = bucket.copy_blob(source_blob, bucket, "uploads/sample_copy.txt")
print("Copied to:", copied_blob.name)

# gsutil equivalent:
# gsutil cp gs://BUCKET_NAME/uploads/sample.txt gs://BUCKET_NAME/uploads/sample_copy.txt


> 🖥️ **Check it in the console:** **Objects** tab → `uploads/sample_copy.txt` should now appear
> alongside the original — both fully independent objects at this point (editing one doesn't
> affect the other).


### 7.2 Rename / Move an Object
GCS has no native rename — `rename_blob` copies to the new name then deletes the original.

In [ ]:
renamed_blob = bucket.rename_blob(copied_blob, "archive/sample_renamed.txt")
print("Renamed/moved to:", renamed_blob.name)


> 🖥️ **Check it in the console:** **Objects** tab → `uploads/sample_copy.txt` should be **gone**,
> and a new `archive/sample_renamed.txt` should appear — visual confirmation that "rename" really
> is copy-then-delete under the hood, not an atomic move.


### 7.3 Delete an Object

In [ ]:
bucket.blob("archive/sample_renamed.txt").delete()
print("Deleted archive/sample_renamed.txt")

# gsutil equivalent: gsutil rm gs://BUCKET_NAME/archive/sample_renamed.txt


> 🖥️ **Check it in the console:** **Objects** tab → `archive/` folder should now be empty or gone
> entirely if it was the only object there — deletion is immediate and (without versioning
> enabled at this point) unrecoverable.


## 8. Object Metadata
### 8.1 Read System Metadata

In [ ]:
blob = bucket.blob("uploads/sample.txt")
blob.reload()   # fetch latest metadata from GCS

print("Size:        ", blob.size)
print("Content-Type:", blob.content_type)
print("MD5 hash:    ", blob.md5_hash)
print("Generation:  ", blob.generation)
print("Updated:     ", blob.updated)


> 🖥️ **Check it in the console:** **Objects** tab → click `uploads/sample.txt` → the **Object
> details** panel on the right shows the exact same size, content-type, and generation number
> just printed — this panel is the console's view of every field a `blob.reload()` fetches.


### 8.2 Set Custom Metadata & Content-Type

In [ ]:
blob.metadata = {"lab-owner": "instructor", "purpose": "gcs-demo"}
blob.content_type = "text/plain"
blob.patch()

print("Custom metadata now:", blob.metadata)


> 🖥️ **Check it in the console:** same **Object details** panel → scroll to **Custom metadata** →
> `lab-owner: instructor` and `purpose: gcs-demo` should now be listed there, editable directly
> from this panel too if you click **Edit metadata**.


## 9. Storage Classes

| Class     | Min. storage duration | Typical access pattern      | Relative cost               |
|-----------|------------------------|------------------------------|------------------------------|
| STANDARD  | none                   | Frequently accessed / hot   | Highest storage, lowest retrieval |
| NEARLINE  | 30 days                | ~Monthly access              | Lower storage, small retrieval fee |
| COLDLINE  | 90 days                | ~Quarterly access            | Lower still, higher retrieval fee |
| ARCHIVE   | 365 days               | Rare / compliance archives   | Lowest storage, highest retrieval fee |

Good talking point for class: storage class is set **per object** or as a **bucket default** — changing
an object's class doesn't move it between locations, it's a metadata-level rewrite.


### 9.1 Change the Bucket's Default Storage Class

In [ ]:
bucket.storage_class = "NEARLINE"
bucket.patch()
print("Bucket default storage class is now:", bucket.storage_class)


> 🖥️ **Check it in the console:** **Buckets > [your bucket] > Configuration** tab → **Default
> storage class** should now read **Nearline** — note this only affects *new* objects going
> forward, not objects already stored, which is exactly why Section 9.2 changes an existing
> object separately.


### 9.2 Change the Storage Class of an Existing Object

In [ ]:
blob = bucket.blob("uploads/sample.txt")
blob.update_storage_class("COLDLINE")

refreshed = bucket.get_blob("uploads/sample.txt")
print("Object storage class is now:", refreshed.storage_class)


> 🖥️ **Check it in the console:** **Objects** tab → click `uploads/sample.txt` → **Object
> details** → **Storage class** should now read **Coldline**, independent of the bucket's default
> — confirming class is genuinely per-object, not just inherited.


## 10. Object Lifecycle Management
### 10.1 Define & Apply Lifecycle Rules
Example policy: move objects to COLDLINE after 30 days, delete anything older than 365 days.

In [ ]:
rules = [
    {
        "action": {"type": "SetStorageClass", "storageClass": "COLDLINE"},
        "condition": {"age": 30},
    },
    {
        "action": {"type": "Delete"},
        "condition": {"age": 365, "isLive": True},
    },
]

bucket.lifecycle_rules = rules
bucket.patch()
print("Lifecycle rules applied.")


> 🖥️ **Check it in the console:** **Buckets > [your bucket] > Lifecycle** tab → the rule set
> ("Set storage class to Coldline after 30 days" / "Delete after 365 days") should be listed here
> in plain English, generated from the JSON rules this cell just applied.


### 10.2 View the Current Lifecycle Configuration

In [ ]:
bucket.reload()
for rule in bucket.lifecycle_rules:
    print(rule)

# gsutil equivalent: gsutil lifecycle get gs://BUCKET_NAME

### 10.3 Remove All Lifecycle Rules

In [ ]:
bucket.clear_lifecycle_rules()
bucket.patch()
print("Lifecycle rules after removal:", bucket.lifecycle_rules)


> 🖥️ **Check it in the console:** **Buckets > [your bucket] > Lifecycle** tab → should now show
> no rules at all — confirming `clear_lifecycle_rules()` removed everything, not just one rule.


## 11. Object Versioning
### 11.1 Enable Versioning

In [ ]:
bucket.versioning_enabled = True
bucket.patch()
print("Versioning enabled:", bucket.versioning_enabled)


> 🖥️ **Check it in the console:** **Buckets > [your bucket] > Protection** tab → **Object
> versioning** should now show **Enabled** — from this point forward, every object write in this
> notebook creates a new generation instead of overwriting.


### 11.2 Create Multiple Versions & List Generations
Each upload to the same object name creates a new **generation** instead of overwriting in place.

In [ ]:
blob = bucket.blob("uploads/versioned.txt")
blob.upload_from_string("version 1")
blob.upload_from_string("version 2")
blob.upload_from_string("version 3")

print("All generations of uploads/versioned.txt:")
for b in client.list_blobs(bucket, prefix="uploads/versioned.txt", versions=True):
    print(" generation:", b.generation, "| size:", b.size, "bytes")


> 🖥️ **Check it in the console:** **Objects** tab → click `uploads/versioned.txt` → a **Live and
> noncurrent versions** (or similarly labeled) toggle/tab shows all 3 generations listed with
> distinct timestamps — the same data this cell just printed, browsable without code.


### 11.3 Delete a Specific Version (generation)

In [ ]:
versions = list(client.list_blobs(bucket, prefix="uploads/versioned.txt", versions=True))
oldest = sorted(versions, key=lambda b: b.generation)[0]

bucket.blob(oldest.name, generation=oldest.generation).delete()
print("Deleted generation:", oldest.generation)


> 🖥️ **Check it in the console:** same object's version list → the oldest generation should now
> be gone, while the two newer ones remain — confirming version deletion is precise, not
> all-or-nothing.


## 12. Access Control — ACLs & IAM
**Teaching point:** fine-grained ACLs and uniform bucket-level access (IAM-only) are mutually exclusive. We enabled uniform access in Section 3.4, so we disable it here to demo legacy ACLs, then show the modern IAM-based approach.

In [ ]:
bucket.iam_configuration.uniform_bucket_level_access_enabled = False
bucket.patch()
print("Uniform bucket-level access disabled - fine-grained ACLs are now usable.")


> 🖥️ **Check it in the console:** **Buckets > [your bucket] > Configuration** tab → **Access
> control** should now read **Fine-grained** instead of **Uniform** — this toggle is exactly what
> the next few ACL-based cells depend on.


### 12.2 Grant Read Access to a Specific User via ACL

In [ ]:
blob = bucket.blob("uploads/sample.txt")
blob.acl.reload()
blob.acl.user("someone@example.com").grant_read()
blob.acl.save()
print("Granted READ on", blob.name, "to someone@example.com")


> 🖥️ **Check it in the console:** **Objects** tab → click `uploads/sample.txt` → **Permissions**
> tab (only visible with fine-grained access enabled) → `someone@example.com` should be listed
> with **Reader** access, granted at the object level specifically.


### 12.3 Make an Object Publicly Readable
**Use with caution in a live demo** — this exposes the object to the entire internet.

In [ ]:
blob.make_public()
print("Public URL:", blob.public_url)


> 🖥️ **Check it in the console:** same object's **Permissions** tab → **allUsers** should now show
> **Reader** — and the **Public access** column back on the Objects list view should now read
> **Public to internet** in place of "Not public."


### 12.4 IAM Policy at the Bucket Level (recommended over ACLs)

In [ ]:
policy = bucket.get_iam_policy(requested_policy_version=3)
policy.version = 3
policy.bindings.append({
    "role": "roles/storage.objectViewer",
    "members": {"group:data-readers@example.com"},
})
bucket.set_iam_policy(policy)
print("IAM policy updated on bucket:", bucket.name)


> 🖥️ **Check it in the console:** **Buckets > [your bucket] > Permissions** tab (bucket-level, not
> object-level) → `data-readers@example.com` should be listed with the **Storage Object Viewer**
> role — this is the IAM-based alternative to the object-level ACL from Section 12.2, applied
> once at the bucket instead of per-object.


### 12.5 View the Current Bucket IAM Policy

In [ ]:
policy = bucket.get_iam_policy(requested_policy_version=3)
for binding in policy.bindings:
    print(binding["role"], "->", binding["members"])

# gsutil equivalent: gsutil iam get gs://BUCKET_NAME


> 🖥️ **Check it in the console:** same bucket **Permissions** tab → every binding printed by this
> cell should have a matching row here, in the console's role-picker UI rather than raw JSON.


## 13. Signed URLs — Temporary, Credential-less Access
**Note for class:** signing requires either a service-account key file or the `roles/iam.serviceAccountTokenCreator` role for impersonation. Plain end-user ADC credentials often can't sign directly — a common point of confusion worth calling out live.

In [ ]:
signed_get_url = blob.generate_signed_url(
    version="v4",
    expiration=timedelta(minutes=15),
    method="GET",
)
print("Signed GET URL (valid 15 min):\n", signed_get_url)


> 🖥️ **Check it in the console:** paste the printed URL into a private/incognito browser window →
> it should load the object's content directly, with no Google sign-in prompt at all — proof the
> signature itself is the credential, valid only until it expires in 15 minutes.


### 13.2 Signed URL for Uploading (PUT)

In [ ]:
upload_blob = bucket.blob("uploads/via_signed_url.txt")
signed_put_url = upload_blob.generate_signed_url(
    version="v4",
    expiration=timedelta(minutes=15),
    method="PUT",
    content_type="text/plain",
)
print("Signed PUT URL (valid 15 min):\n", signed_put_url)


> 🖥️ **Check it in the console:** there's no dedicated console UI for testing a signed PUT URL —
> the practical way to verify it is a `curl -X PUT --upload-file localfile.txt "SIGNED_URL"` from
> a terminal, then confirming the object appears in the **Objects** tab afterward.


## 14. CORS Configuration
Needed when a browser-based app on a different origin needs to call GCS directly (e.g. direct-to-bucket uploads).

In [ ]:
bucket.cors = [
    {
        "origin": ["https://example.com"],
        "method": ["GET", "HEAD"],
        "responseHeader": ["Content-Type"],
        "maxAgeSeconds": 3600,
    }
]
bucket.patch()
print("CORS configuration applied:", bucket.cors)


> 🖥️ **Check it in the console:** **Buckets > [your bucket] > Configuration** tab → a **CORS**
> section should list the origin, methods, and headers just configured — most useful to actually
> exercise from a real browser-based app on that origin, not from this notebook.


## 15. Compose Objects (server-side concatenation)
Combines multiple objects into one **without downloading and re-uploading** — useful after parallel chunked uploads.

In [ ]:
part1 = bucket.blob("compose/part1.txt")
part1.upload_from_string("Hello, ")

part2 = bucket.blob("compose/part2.txt")
part2.upload_from_string("World!")

composed = bucket.blob("compose/combined.txt")
composed.compose([part1, part2])

print("Composed object content:", composed.download_as_text())


> 🖥️ **Check it in the console:** **Objects** tab → `compose/combined.txt` should be listed with a
> size equal to the sum of `part1.txt` + `part2.txt` — and critically, this object's **generation**
> and upload timestamp reflect a single compose operation, not two separate uploads.


## 16. Cleanup — Delete All Objects & the Bucket
Run this **last**, once every demo above has been shown, so the class sees teardown too.

In [ ]:
# delete every live object AND every archived version
for b in client.list_blobs(bucket, versions=True):
    bucket.blob(b.name, generation=b.generation).delete()

bucket.delete()
print(f"Bucket {BUCKET_NAME} and all contents deleted.")

# gsutil equivalent:
# gsutil rm -r gs://BUCKET_NAME


### 16.1 Final Verification


In [ ]:

# Confirm the bucket is really gone -- this should raise NotFound if cleanup succeeded
try:
    client.get_bucket(BUCKET_NAME)
    print("WARNING: bucket still exists — cleanup may not have completed.")
except Exception as e:
    print("Confirmed: bucket no longer exists.", type(e).__name__)



> 🖥️ **Final console check:** **Cloud Storage > Buckets** → your bucket (`gcs-training-demo-...`)
> should no longer be listed at all under your project — confirming every object, every version,
> and the bucket itself were all removed.
